In [11]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split

In [2]:
# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
EVENTS_PATH = "../data/reconstructed_events_v3.csv"  # adjust to your actual path
OUTPUT_PATH = "../data/model_ready_cart_sessions_final.csv"
ABANDONMENT_WINDOW_HOURS = 24
RANDOM_STATE = 42

In [3]:
events = pd.read_csv(EVENTS_PATH)
events["event_time"] = pd.to_datetime(events["event_time"], utc=True)
events = events.sort_values(["user_id", "event_time"]).reset_index(drop=True)
 
first_cart = (
    events[events["event_type"] == "cart"]
    .groupby("reconstructed_session_id")
    .agg(user_id=("user_id", "first"), cart_time=("event_time", "min"))
    .reset_index()
)
 
purchases = events.loc[
    events["event_type"] == "purchase", ["user_id", "event_time"]
].rename(columns={"event_time": "purchase_time"})
purchases = purchases.sort_values("purchase_time").reset_index(drop=True)
 
cart_lookup = first_cart.sort_values("cart_time").reset_index(drop=True)

In [4]:
# THE FIX: match each cart to the user's NEXT purchase, any session, using
# a time-ordered merge_asof on user_id (not on reconstructed_session_id).
next_purchase = pd.merge_asof(
    cart_lookup,
    purchases,
    left_on="cart_time",
    right_on="purchase_time",
    by="user_id",
    direction="forward",
)
next_purchase["hours_to_purchase"] = (
    next_purchase["purchase_time"] - next_purchase["cart_time"]
).dt.total_seconds() / 3600
next_purchase["is_abandoned"] = (
    next_purchase["purchase_time"].isna()
    | (next_purchase["hours_to_purchase"] > ABANDONMENT_WINDOW_HOURS)
).astype(int)
 
print("=== Target distribution (corrected, cross-session) ===")
print(next_purchase["is_abandoned"].value_counts())
print(next_purchase["is_abandoned"].value_counts(normalize=True) * 100)

=== Target distribution (corrected, cross-session) ===
is_abandoned
0    23304
1    19549
Name: count, dtype: int64
is_abandoned
0    54.381257
1    45.618743
Name: proportion, dtype: float64


In [5]:
# ==================================================================
# PART 2: SESSION-LEVEL HISTORY FEATURES
# ==================================================================
 
session_cart_flag = (
    events.groupby("reconstructed_session_id")
    .agg(
        user_id=("user_id", "first"),
        session_start=("event_time", "min"),
        session_has_cart=("event_type", lambda x: int((x == "cart").any())),
    )
    .reset_index()
    .sort_values(["user_id", "session_start"])
)
session_cart_flag["previous_cart_session_count"] = (
    session_cart_flag.groupby("user_id")["session_has_cart"]
    .cumsum()
    .shift(1)
    .fillna(0)
    .astype(int)
)
 
session_purchase_flag = (
    events.groupby("reconstructed_session_id")["event_type"]
    .apply(lambda x: int((x == "purchase").any()))
    .reset_index(name="session_has_purchase")
)
 
session_history = session_cart_flag.merge(session_purchase_flag, on="reconstructed_session_id")
session_history["previous_purchase_count"] = (
    session_history.groupby("user_id")["session_has_purchase"]
    .cumsum()
    .shift(1)
    .fillna(0)
    .astype(int)
)
session_history["previous_session_count"] = session_history.groupby("user_id").cumcount()
session_history["has_previous_purchase"] = (
    session_history["previous_purchase_count"] > 0
).astype(int)
session_history["user_purchase_rate"] = (
    session_history["previous_purchase_count"]
    / session_history["previous_session_count"].replace(0, np.nan)
).fillna(0)

In [6]:
# ==================================================================
# PART 3: BEHAVIORAL FEATURES (PRICE, VIEWS, DURATION, DIVERSITY)
# ==================================================================
 
events_with_cart_time = events.merge(
    first_cart[["reconstructed_session_id", "cart_time"]], on="reconstructed_session_id"
)
events_before_cart = events_with_cart_time[
    events_with_cart_time["event_time"] <= events_with_cart_time["cart_time"]
]
 
views_before_cart = (
    events_before_cart[events_before_cart["event_type"] == "view"]
    .groupby("reconstructed_session_id")
    .size()
    .reset_index(name="views_before_cart")
)
 
price_summary = (
    events_before_cart.groupby("reconstructed_session_id")
    .agg(
        avg_price_before_cart=("price", "mean"),
        min_price=("price", "min"),
        max_price=("price", "max"),
    )
    .reset_index()
)
price_summary["price_range_before_cart"] = price_summary["max_price"] - price_summary["min_price"]
 
brand_cat_summary = (
    events_before_cart.groupby("reconstructed_session_id")
    .agg(
        unique_brands_before_cart=("brand", "nunique"),
        unique_categories_before_cart=("category_code", "nunique"),
        unique_products_before_cart=("product_id", "nunique"),
        total_interactions_before_cart=("event_type", "size"),
    )
    .reset_index()
)
 
duration_summary = (
    events_before_cart.groupby("reconstructed_session_id")
    .agg(session_start=("event_time", "min"))
    .reset_index()
    .merge(first_cart[["reconstructed_session_id", "cart_time"]], on="reconstructed_session_id")
)
duration_summary["duration_before_cart"] = (
    duration_summary["cart_time"] - duration_summary["session_start"]
).dt.total_seconds() / 60
 
prev_purchase = pd.merge_asof(
    cart_lookup.sort_values("cart_time").reset_index(drop=True),
    purchases.sort_values("purchase_time").reset_index(drop=True),
    left_on="cart_time",
    right_on="purchase_time",
    by="user_id",
    direction="backward",
    allow_exact_matches=False,
)
prev_purchase["days_since_last_purchase"] = (
    (prev_purchase["cart_time"] - prev_purchase["purchase_time"]).dt.total_seconds() / 86400
).fillna(-1)

In [7]:
# ==================================================================
# PART 4: NEW FEATURES -- HISTORICAL CATEGORY/PRODUCT SIGNALS
# ==================================================================
 
first_cart_events = (
    events[events["event_type"] == "cart"]
    .sort_values("event_time")
    .drop_duplicates(subset="reconstructed_session_id", keep="first")[
        ["reconstructed_session_id", "product_id", "category_code", "brand"]
    ]
)
 
cart_context = first_cart.merge(first_cart_events, on="reconstructed_session_id", how="left")
cart_context = cart_context.merge(
    next_purchase[["reconstructed_session_id", "is_abandoned"]], on="reconstructed_session_id"
)
cart_context["category_code"] = cart_context["category_code"].fillna("unknown")
cart_context = cart_context.sort_values("cart_time").reset_index(drop=True)
 
# Leak-free historical category abandonment rate (only uses PRIOR sessions in
# the same category, via a shift-then-cumsum pattern -- current row excluded)
cart_context["is_abandoned_shifted"] = cart_context.groupby("category_code")["is_abandoned"].shift(1)
cart_context["cat_prior_abandon_sum"] = (
    cart_context.groupby("category_code")["is_abandoned_shifted"].cumsum()
)
cart_context["cat_prior_count"] = cart_context.groupby("category_code").cumcount()
overall_abandon_rate = cart_context["is_abandoned"].mean()
cart_context["category_historical_abandon_rate"] = (
    cart_context["cat_prior_abandon_sum"] / cart_context["cat_prior_count"].replace(0, np.nan)
).fillna(overall_abandon_rate)
 
cart_context["product_prior_cart_count"] = cart_context.groupby("product_id").cumcount()
 
# Within-session, product-specific behavior
cart_item_info = cart_context[["reconstructed_session_id", "product_id", "category_code"]].rename(
    columns={"product_id": "cart_product_id", "category_code": "cart_category_code"}
)
events_before_cart_ctx = events_before_cart.merge(cart_item_info, on="reconstructed_session_id")
 
same_product_views = events_before_cart_ctx[
    (events_before_cart_ctx["event_type"] == "view")
    & (events_before_cart_ctx["product_id"] == events_before_cart_ctx["cart_product_id"])
]
product_repeat_views = (
    same_product_views.groupby("reconstructed_session_id")
    .size()
    .reset_index(name="product_repeat_views_before_cart")
)
 
first_view_of_cart_product = (
    same_product_views.groupby("reconstructed_session_id")["event_time"]
    .min()
    .reset_index(name="first_view_time")
)
view_to_cart = first_view_of_cart_product.merge(
    first_cart[["reconstructed_session_id", "cart_time"]], on="reconstructed_session_id"
)
view_to_cart["view_to_cart_minutes"] = (
    view_to_cart["cart_time"] - view_to_cart["first_view_time"]
).dt.total_seconds() / 60
 
views_only_ctx = events_before_cart_ctx[events_before_cart_ctx["event_type"] == "view"]
category_focus = (
    views_only_ctx.assign(same_category=lambda d: d["category_code"] == d["cart_category_code"])
    .groupby("reconstructed_session_id")["same_category"]
    .mean()
    .reset_index(name="category_view_concentration")
)

In [8]:
# ==================================================================
# PART 5: ASSEMBLE FINAL MODEL-READY DATASET
# ==================================================================
 
model_df = first_cart[["reconstructed_session_id", "cart_time"]].copy()
model_df = model_df.merge(
    next_purchase[["reconstructed_session_id", "is_abandoned"]], on="reconstructed_session_id"
)
model_df = model_df.merge(
    session_history[
        [
            "reconstructed_session_id",
            "previous_session_count",
            "previous_purchase_count",
            "has_previous_purchase",
            "user_purchase_rate",
            "previous_cart_session_count",
        ]
    ],
    on="reconstructed_session_id",
)
model_df = model_df.merge(views_before_cart, on="reconstructed_session_id", how="left")
model_df = model_df.merge(
    price_summary[["reconstructed_session_id", "avg_price_before_cart", "price_range_before_cart"]],
    on="reconstructed_session_id",
    how="left",
)
model_df = model_df.merge(brand_cat_summary, on="reconstructed_session_id", how="left")
model_df = model_df.merge(
    duration_summary[["reconstructed_session_id", "duration_before_cart"]],
    on="reconstructed_session_id",
    how="left",
)
model_df = model_df.merge(
    prev_purchase[["reconstructed_session_id", "days_since_last_purchase"]],
    on="reconstructed_session_id",
    how="left",
)
model_df = model_df.merge(
    cart_context[["reconstructed_session_id", "category_historical_abandon_rate", "product_prior_cart_count"]],
    on="reconstructed_session_id",
    how="left",
)
model_df = model_df.merge(product_repeat_views, on="reconstructed_session_id", how="left")
model_df = model_df.merge(
    view_to_cart[["reconstructed_session_id", "view_to_cart_minutes"]],
    on="reconstructed_session_id",
    how="left",
)
model_df = model_df.merge(category_focus, on="reconstructed_session_id", how="left")
 
model_df["views_before_cart"] = model_df["views_before_cart"].fillna(0).astype(int)
model_df["cart_to_view_ratio"] = 1 / model_df["views_before_cart"].replace(0, 1)
model_df["no_view_before_cart"] = (model_df["views_before_cart"] == 0).astype(int)
model_df["cart_hour"] = model_df["cart_time"].dt.hour
model_df["cart_day_of_week"] = model_df["cart_time"].dt.dayofweek
model_df["cart_is_weekend"] = (model_df["cart_day_of_week"] >= 5).astype(int)
model_df["never_viewed_cart_item_before"] = model_df["product_repeat_views_before_cart"].isna().astype(int)
model_df["product_repeat_views_before_cart"] = model_df["product_repeat_views_before_cart"].fillna(0)
model_df["view_to_cart_minutes"] = model_df["view_to_cart_minutes"].fillna(-1)
model_df["category_view_concentration"] = model_df["category_view_concentration"].fillna(0)
 
model_df = model_df.drop(columns=["cart_time"]).fillna(0)
 
model_df.to_csv(OUTPUT_PATH, index=False)
print("\n=== Final dataset ===")
print("Shape:", model_df.shape)
print("Missing values:", model_df.isnull().sum().sum())
print("Saved to:", OUTPUT_PATH)


=== Final dataset ===
Shape: (42853, 27)
Missing values: 0
Saved to: ../data/model_ready_cart_sessions_final.csv


In [9]:
# ==================================================================
# PART 6: TRAIN AND EVALUATE THE FINAL MODEL (GOAL 1)
# ==================================================================
 
X = model_df.drop(columns=["reconstructed_session_id", "is_abandoned"])
y = model_df["is_abandoned"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
 
baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
print("\nBaseline accuracy:", accuracy_score(y_test, baseline.predict(X_test)))
 
model = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]
 
print("\n=== Final model: test set results ===")
print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall   :", recall_score(y_test, pred))
print("F1 Score :", f1_score(y_test, pred))
print("ROC-AUC  :", roc_auc_score(y_test, proba))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))


Baseline accuracy: 0.5438105238595263

=== Final model: test set results ===
Accuracy : 0.588612763971532
Precision: 0.5608750792644261
Recall   : 0.45242966751918157
F1 Score : 0.5008493771234428
ROC-AUC  : 0.6186939182452642
Confusion matrix:
 [[3276 1385]
 [2141 1769]]


In [10]:
# ==================================================================
# PART 7: SHAP EXPLAINABILITY (GOAL 2)
# ==================================================================
 
import shap
import matplotlib.pyplot as plt
 
explainer = shap.TreeExplainer(model)
 
sample_size = min(2000, X_test.shape[0])
X_sample = X_test.sample(n=sample_size, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(X_sample)
 
global_importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=X_sample.columns
).sort_values(ascending=False)
 
print("\n=== Global SHAP feature importance ===")
print(global_importance)
 
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("shap_summary_bar.png", dpi=150)
plt.close()
print("\nSaved SHAP summary plot to shap_summary_bar.png")
 
# Individual case study: pick a session the model is confident is abandoned
proba_sample = model.predict_proba(X_sample)[:, 1]
top_abandoned_idx = np.argmax(proba_sample)
print(f"\n=== Individual case study ===")
print("Session index in sample:", top_abandoned_idx)
print("Predicted abandonment probability:", proba_sample[top_abandoned_idx])
 
case_shap = pd.Series(
    shap_values[top_abandoned_idx], index=X_sample.columns
).sort_values(key=abs, ascending=False)
case_values = X_sample.iloc[top_abandoned_idx]
 
print("\nTop SHAP contributions for this session:")
for feat in case_shap.index[:10]:
    print(f"  {feat:35s} value={case_values[feat]:>12.3f}  shap={case_shap[feat]:+.4f}")
 
print(
    "\nREMINDER: SHAP values show what CONTRIBUTED to this prediction, not "
    "what CAUSED the abandonment. Use wording like 'associated with' or "
    "'contributed to the model's prediction' when reporting these findings."
)


=== Global SHAP feature importance ===
days_since_last_purchase            0.148571
avg_price_before_cart               0.131579
category_historical_abandon_rate    0.114107
product_prior_cart_count            0.080482
view_to_cart_minutes                0.067160
cart_hour                           0.046439
duration_before_cart                0.033729
price_range_before_cart             0.021083
user_purchase_rate                  0.015772
previous_session_count              0.015413
total_interactions_before_cart      0.014893
previous_purchase_count             0.008020
previous_cart_session_count         0.006941
product_repeat_views_before_cart    0.006311
category_view_concentration         0.005578
cart_day_of_week                    0.003436
unique_categories_before_cart       0.003285
unique_products_before_cart         0.003196
has_previous_purchase               0.002963
unique_brands_before_cart           0.002198
cart_is_weekend                     0.002019
views_before_ca

In [12]:
# Drop features SHAP showed have zero/near-zero importance
low_impact_features = [
    "no_view_before_cart",
    "never_viewed_cart_item_before",
    "cart_to_view_ratio",
    "views_before_cart",
]
X_clean = X.drop(columns=low_impact_features)

# Fresh, consistent train/test split on the cleaned feature set
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clean, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Clean X_train shape:", X_train_c.shape)
print("Clean X_test shape:", X_test_c.shape)

# Proper hyperparameter tuning on the cleaned, consistent split
param_distributions = {
    "n_estimators": randint(100, 400),
    "learning_rate": uniform(0.01, 0.15),
    "max_depth": randint(2, 6),
    "min_samples_leaf": randint(5, 50),
    "subsample": uniform(0.6, 0.4),
}

search = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_distributions=param_distributions,
    n_iter=30,
    scoring="roc_auc",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train_c, y_train_c)

print("\nBest parameters:", search.best_params_)
print("Best CV ROC-AUC:", search.best_score_)

final_model = search.best_estimator_
final_pred = final_model.predict(X_test_c)
final_proba = final_model.predict_proba(X_test_c)[:, 1]

print("\n=== FINAL tuned + cleaned model: test set results ===")
print("Accuracy :", accuracy_score(y_test_c, final_pred))
print("Precision:", precision_score(y_test_c, final_pred))
print("Recall   :", recall_score(y_test_c, final_pred))
print("F1 Score :", f1_score(y_test_c, final_pred))
print("ROC-AUC  :", roc_auc_score(y_test_c, final_proba))
print("Confusion matrix:\n", confusion_matrix(y_test_c, final_pred))
print("Confusion matrix total:", confusion_matrix(y_test_c, final_pred).sum(), "(should be", len(y_test_c), ")")

Clean X_train shape: (34282, 21)
Clean X_test shape: (8571, 21)
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters: {'learning_rate': np.float64(0.019533752542903543), 'max_depth': 4, 'min_samples_leaf': 28, 'n_estimators': 392, 'subsample': np.float64(0.8918424713352255)}
Best CV ROC-AUC: 0.6210010114008527

=== FINAL tuned + cleaned model: test set results ===
Accuracy : 0.5910628864776573
Precision: 0.5624421831637373
Recall   : 0.46649616368286445
F1 Score : 0.5099958059555432
ROC-AUC  : 0.6204328950407995
Confusion matrix:
 [[3242 1419]
 [2086 1824]]
Confusion matrix total: 8571 (should be 8571 )
